# Refi-Map Generator

Upload your geocoded Excel file and get back a **single `refi_map.html`** you can open directly in Chrome.

**Expected columns:** Prop ID, Property Type, Description, Land Use Class, Full Address (or Address), City, Vendor Company, Vendor Director, Purchaser Company, Purchaser Director, Subdivision, Site Area, Site Units, Sale Price, Sale Date, Unit Price, Unit Price Measure, Cap Rate, Total Units, Year Built, latitude, longitude

**Run each cell in order** (Shift+Enter or click ▶)

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q pandas openpyxl requests

In [ ]:
# ── Step 2: Upload Excel file ─────────────────────────────────────────────────
import pandas as pd
from google.colab import files

print('Upload your geocoded Excel file:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f'\nLoaded {len(df)} rows, {len(df.columns)} columns')
df.head(3)

In [ ]:
# ── Step 3: Process all rows ──────────────────────────────────────────────────
import json
from datetime import date
from dateutil.relativedelta import relativedelta

TODAY = date.today()

def clean(val, fallback=''):
    try:
        if pd.isna(val): return fallback
    except (TypeError, ValueError):
        pass
    return str(val).strip()

def fmt_price(val):
    try: return '$' + format(int(float(val)), ',')
    except Exception: return 'N/A'

def fmt_ppu(price_val, measure_val):
    try:
        v = float(price_val)
        measure = clean(measure_val)
        price_str = '$' + format(int(v), ',') if v >= 1 else '$' + f'{v:.2f}'
        return price_str + '/' + measure if measure else price_str
    except Exception: return 'N/A'

def fmt_cap(val):
    try:
        v = float(val)
        if 0 < v < 1: v *= 100
        return f'{v:.1f}%'
    except Exception: return '0.0%'

def fmt_date_display(dt):
    try: return pd.to_datetime(dt).strftime('%b %Y')
    except Exception: return ''

def fmt_date_iso(dt):
    try: return pd.to_datetime(dt).strftime('%Y-%m-%d')
    except Exception: return ''

def fmt_year(val):
    try:
        v = int(float(val))
        return str(v) if v > 0 else ''
    except Exception: return ''

def fmt_area(val):
    try:
        if pd.isna(val): return ''
        v = float(val)
        if v == 0: return ''
        return str(int(v)) if v == int(v) else f'{v:.4f}'.rstrip('0').rstrip('.')
    except Exception: return ''

def calc_refi(dt):
    try:
        sale = pd.to_datetime(dt).date()
        refi = sale + relativedelta(years=5)
        months_out = (refi.year - TODAY.year) * 12 + (refi.month - TODAY.month)
        is_refi = 0 <= months_out <= 18
        months_str = f'~{months_out} months' if months_out >= 0 else 'Past due'
        return refi.strftime('%b %Y'), months_str, is_refi
    except Exception: return '', '', False

props = []
skipped = []

for idx, row in df.iterrows():
    lat = row.get('latitude')
    lng = row.get('longitude')
    if pd.isna(lat) or pd.isna(lng):
        addr = row.get('Full Address', row.get('Address', 'unknown'))
        skipped.append('Row ' + str(idx + 2) + ': ' + str(addr) + ' — no coordinates')
        continue

    sale_dt = row.get('Sale Date', '')
    refi_date, months_out_str, is_refi = calc_refi(sale_dt)
    units_raw = row.get('Total Units')
    total_units = str(int(float(units_raw))) if pd.notna(units_raw) else '0'

    props.append({
        'address':     clean(row.get('Full Address') or row.get('Address')),
        'saleDate':    fmt_date_display(sale_dt),
        'saleDateIso': fmt_date_iso(sale_dt),
        'salePrice':   fmt_price(row.get('Sale Price')),
        'totalUnits':  total_units,
        'ppu':         fmt_ppu(row.get('Unit Price'), row.get('Unit Price Measure')),
        'capRate':     fmt_cap(row.get('Cap Rate')),
        'yearBuilt':   fmt_year(row.get('Year Built')),
        'buyer':       clean(row.get('Purchaser Company')),
        'buyerDir':    clean(row.get('Purchaser Director')),
        'seller':      clean(row.get('Vendor Company')),
        'sellerDir':   clean(row.get('Vendor Director')),
        'refiDate':    refi_date,
        'monthsOut':   months_out_str,
        'isRefi':      is_refi,
        'landUse':     clean(row.get('Land Use Class')),
        'propType':    clean(row.get('Property Type')),
        'description': clean(row.get('Description')),
        'subdivision': clean(row.get('Subdivision')),
        'siteArea':    fmt_area(row.get('Site Area')),
        'siteUnits':   clean(row.get('Site Units')),
        'propId':      str(int(float(row.get('Prop ID', 0)))),
        'lat':         round(float(lat), 8),
        'lng':         round(float(lng), 8),
    })

print('✅  ' + str(len(props)) + ' properties processed')
if skipped:
    print('⚠️   ' + str(len(skipped)) + ' rows skipped (no coordinates):')
    for s in skipped: print('   ' + s)

In [ ]:
# ── Step 4: Build inline data script ─────────────────────────────────────────

dates    = [p['saleDateIso'] for p in props if p['saleDateIso']]
date_min = min(dates) if dates else ''
date_max = max(dates) if dates else ''

def jsv(v):     return json.dumps(v)
def jsbool(v):  return 'true' if v else 'false'

out = [
    '// Edmonton Multifamily Sales Data',
    '// Generated: ' + str(TODAY),
    '//',
    'const DATE_MIN = ' + jsv(date_min) + ';',
    'const DATE_MAX = ' + jsv(date_max) + ';',
    '',
    'const allProps = [',
]

for p in props:
    out += [
        '  {',
        '    address: '      + jsv(p['address']) + ',',
        '    saleDate: '     + jsv(p['saleDate']) + ', saleDateIso: ' + jsv(p['saleDateIso']) + ',',
        '    salePrice: '    + jsv(p['salePrice']) + ', totalUnits: ' + jsv(p['totalUnits']) + ', ppu: ' + jsv(p['ppu']) + ', capRate: ' + jsv(p['capRate']) + ',',
        '    yearBuilt: '    + jsv(p['yearBuilt']) + ',',
        '    buyer: '        + jsv(p['buyer']) + ', buyerDir: ' + jsv(p['buyerDir']) + ',',
        '    seller: '       + jsv(p['seller']) + ', sellerDir: ' + jsv(p['sellerDir']) + ',',
        '    refiDate: '     + jsv(p['refiDate']) + ', monthsOut: ' + jsv(p['monthsOut']) + ', isRefi: ' + jsbool(p['isRefi']) + ',',
    ]
    if p['landUse']:      out.append('    landUse: '     + jsv(p['landUse']) + ',')
    if p['propType']:     out.append('    propType: '    + jsv(p['propType']) + ',')
    if p['description']:  out.append('    description: ' + jsv(p['description']) + ',')
    if p['subdivision']:  out.append('    subdivision: ' + jsv(p['subdivision']) + ',')
    if p['siteArea']:     out.append('    siteArea: '    + jsv(p['siteArea']) + ', siteUnits: ' + jsv(p['siteUnits']) + ',')
    if p['propId']:       out.append('    propId: '      + jsv(p['propId']) + ',')
    out += [
        '    lat: ' + str(p['lat']) + ', lng: ' + str(p['lng']),
        '  },',
    ]

out.append('];')
data_script = '<script>\n' + '\n'.join(out) + '\n</script>'
print('✅  Data script ready (' + str(len(props)) + ' records, ' + date_min + ' → ' + date_max + ')')

In [ ]:
# ── Step 5: Fetch HTML template from GitHub ───────────────────────────────────
# If the repo is private or the fetch fails, you will be prompted to upload
# index.html manually (download it from your GitHub repo first).

import requests

GITHUB_RAW = 'https://raw.githubusercontent.com/neilmah12/Refi-Map/main/index.html'

html_template = None
try:
    r = requests.get(GITHUB_RAW, timeout=10)
    r.raise_for_status()
    html_template = r.text
    print('✅  Fetched index.html from GitHub')
except Exception as e:
    print('⚠️  Could not fetch from GitHub: ' + str(e))
    print('\nPlease upload index.html from your Refi-Map repo:')
    uploaded_html = files.upload()
    html_fname = list(uploaded_html.keys())[0]
    with open(html_fname, encoding='utf-8') as f:
        html_template = f.read()
    print('✅  index.html uploaded')

In [ ]:
# ── Step 6: Inject data and download standalone HTML ─────────────────────────

# Replace the external script reference with the inline data
PLACEHOLDER = '<script src="properties.js"></script>'
if PLACEHOLDER not in html_template:
    # Fallback: look for the old data/properties.js reference
    PLACEHOLDER = '<script src="data/properties.js"></script>'

if PLACEHOLDER not in html_template:
    print('❌  Could not find the data script tag in index.html.')
    print('    Make sure index.html contains: <script src="properties.js"></script>')
else:
    final_html = html_template.replace(PLACEHOLDER, data_script)

    with open('refi_map.html', 'w', encoding='utf-8') as f:
        f.write(final_html)

    print('✅  refi_map.html ready — ' + str(len(props)) + ' properties embedded')
    print('\nDownloading now...')
    files.download('refi_map.html')
    print('Done! Open refi_map.html in Chrome to view the map.')

## Notes

- **Output** — `refi_map.html` is fully self-contained. No other files needed — just open it in Chrome.
- **Updating the map** — re-run the notebook with a new Excel export to get a fresh HTML file.
- **Cap Rate** — handles both decimal (`0.055`) and percentage (`5.5`) format automatically.
- **Land records** — shown as grey pins on the map. The colour scale (blue→red) only uses building PPU so land transactions don't skew it.
- **Refi window** — recalculated from today's date each time you run the notebook.
- **Private repo** — if Step 5 fails to fetch from GitHub, just download `index.html` from your repo and upload it when prompted.